# 01. Carga, integración y limpieza de datos

## Presentación de la sección
En esta etapa se muestra como se integran los años 2022, 2023 y 2024 a partir de los archivos originales, y como esa integración desemboca en la base maestra final utilizada en el análisis y en el dashboard.


In [73]:
import importlib.util

requeridas = ['pandas', 'numpy']
faltantes = [pkg for pkg in requeridas if importlib.util.find_spec(pkg) is None]
if faltantes:
    raise ModuleNotFoundError(f'Faltan librerias en este kernel: {faltantes}. Instale requirements.txt o seleccione otro interprete.')
print('Entorno verificado correctamente.')


Entorno verificado correctamente.


## Librerias de trabajo


In [74]:
from pathlib import Path
import unicodedata
import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'data').exists():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / 'data').exists():
    PROJECT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError('No se encontro la carpeta data del proyecto.')

DATA_DIR = PROJECT_DIR / 'data'
MASTER_FILE = DATA_DIR / 'contrataciones_peru_2022_2024_maestro.csv'
PERIODOS = [2022, 2023, 2024]

print('Proyecto   :', PROJECT_DIR)
print('Datos      :', DATA_DIR)
print('Base final :', MASTER_FILE)


Proyecto   : c:\Users\lenovo\Desktop\PYTHON\SESIONES ASESOR\DATOS OECE
Datos      : c:\Users\lenovo\Desktop\PYTHON\SESIONES ASESOR\DATOS OECE\data
Base final : c:\Users\lenovo\Desktop\PYTHON\SESIONES ASESOR\DATOS OECE\data\contrataciones_peru_2022_2024_maestro.csv


## Integración de los archivos `main.csv` por año

El archivo `main.csv` es la mejor fuente para mostrar la unión de los años porque conserva el identificador `ocid` y varias variables base del proceso de contratación.


In [75]:
def apilar_main():
    partes = []
    for periodo in PERIODOS:
        ruta = DATA_DIR / str(periodo) / 'main.csv'
        temp = pd.read_csv(ruta, low_memory=False)
        temp['año'] = periodo
        partes.append(temp)
    return pd.concat(partes, ignore_index=True)

main = apilar_main()
print('Dimensión de la tabla main integrada:', main.shape)


Dimensión de la tabla main integrada: (224596, 39)


## Vista preliminar de la integración

Se muestran algunas columnas clave para evidenciar que la unión de los tres años se realiza correctamente.


In [76]:
columnas_mostrar = [
    'ocid', 'anio', 'buyer_name', 'tender_mainProcurementCategory',
    'tender_procurementMethod', 'tender_numberOfTenderers', 'tender_value_amount_PEN'
]
columnas_mostrar = [c for c in columnas_mostrar if c in main.columns]
main[columnas_mostrar].head()


,ocid,buyer_name,tender_mainProcurementCategory,tender_procurementMethod,tender_numberOfTenderers,tender_value_amount_PEN
0,ocds-dgv273-seacev3-810078,GOBIERNO REGIONAL DE AREQUIPA Sede Central,goods,open,1.0,43710.00
1,ocds-dgv273-seacev3-2022-200276-452,ORGANISMO DE EVALUACION Y FISCALIZACION AMBIENTAL,services,open,1.0,30000.00
2,ocds-dgv273-seacev3-2022-1221-4,MUNICIPALIDAD DISTRITAL DE LA CUESTA,works,open,5.0,51893.73
3,ocds-dgv273-seacev3-802121,GOBIERNO REGIONAL DE MADRE DE DIOS - HOSPITAL ...,goods,open,NaN,0.00
4,ocds-dgv273-seacev3-804479,GOBIERNO REGIONAL DE CAJAMARCA - UNIDAD DE GES...,goods,open,6.0,144198.20


## Integración de proveedores ganadores

Además del archivo principal, se consolida la tabla `awards_suppliers.csv` para disponer del proveedor ganador asociado a cada proceso.


In [77]:
def apilar_proveedores():
    partes = []
    for periodo in PERIODOS:
        ruta = DATA_DIR / str(periodo) / 'awards_suppliers.csv'
        temp = pd.read_csv(ruta, low_memory=False)
        temp['anio'] = periodo
        partes.append(temp)
    return pd.concat(partes, ignore_index=True)

suppliers = apilar_proveedores()
print('Dimensión de proveedores integrados:', suppliers.shape)


Dimensión de proveedores integrados: (184176, 6)


## Carga de la base maestra final

La versión final del proyecto trabaja con un único archivo consolidado: `contrataciones_peru_2022_2024_maestro.csv`. Esta base ya resume la limpieza y homologación necesarias para el resto del análisis.


In [78]:
df = pd.read_csv(MASTER_FILE, low_memory=False)
def obtener_columna_anio(dataframe):
    for columna in dataframe.columns:
        normal = unicodedata.normalize('NFKD', str(columna)).encode('ascii', 'ignore').decode('ascii').lower()
        if normal in ('anio', 'ano'):
            return columna
    raise KeyError('No se encontro una columna de año en la base.')

col_anio = obtener_columna_anio(df)
print('Dimensión de la base maestra:', df.shape)


Dimensión de la base maestra: (224596, 28)


## Limpieza mínima para la exposición

En este bloque se verifican tipos, se rellenan valores textuales faltantes y se validan las columnas principales que seran utilizadas en visualizaciones y KPIs.


In [79]:
for columna in ['categoria', 'metodo_simple', 'departamento', 'entidad_compradora', 'proveedor_ganador']:
    if columna in df.columns:
        df[columna] = df[columna].fillna('No especificado').astype(str).str.strip()

df['monto_adjudicado'] = pd.to_numeric(df['monto_adjudicado'], errors='coerce').fillna(0)
df['monto_MM'] = pd.to_numeric(df['monto_MM'], errors='coerce').fillna(df['monto_adjudicado'] / 1_000_000)
df['n_postores'] = pd.to_numeric(df['n_postores'], errors='coerce').fillna(0)
df['un_solo_postor'] = df['n_postores'].eq(1)


## Validación final de la base maestra

La siguiente verificación resume la dimensión de la base, los periodos presentes y la distribución general por categoria.


In [80]:
print('Filas               :', df.shape[0])
print('Columnas            :', df.shape[1])
print('OCID únicos         :', df['ocid'].nunique())
print('Periodos presentes  :', sorted(df[col_anio].dropna().unique().tolist()))
print('Categorías          :', df['categoria'].value_counts(dropna=False).to_dict())
df.head()


Filas               : 224596
Columnas            : 28
OCID únicos         : 224596
Periodos presentes  : [2022, 2023, 2024]
Categorías          : {'Bienes': 106080, 'Servicios': 94777, 'Obras': 23739}


,date,ocid,publishedDate,buyer_name,tender_title,tender_mainProcurementCategory,tender_procurementMethod,tender_datePublished,tender_numberOfTenderers,tender_value_amount_PEN,...,clasificacion,proveedor_ganador,fecha_proceso,mes,semana,categoria,metodo_simple,n_postores,un_solo_postor,monto_MM
0,2022-05-01 00:00:00-05:00,ocds-dgv273-seacev3-810078,2023-12-06 05:31:55.496629-05:00,GOBIERNO REGIONAL DE AREQUIPA Sede Central,AS-SM-16-2022-GRA-2,goods,open,2022-05-25 18:08:00-05:00,1.0,43710.00,...,Sin dato,Sin dato,2022-05-25 18:08:00,2022-05,21,Bienes,Competitivo,1.0,True,0.00
1,2022-05-01 00:00:00-05:00,ocds-dgv273-seacev3-2022-200276-452,2023-12-06 05:26:45.870072-05:00,ORGANISMO DE EVALUACION Y FISCALIZACION AMBIENTAL,RES-PROC-427-2022-OEFA-CTER-1,services,open,2022-05-04 15:19:00-05:00,1.0,30000.00,...,SERVICIO DE SEGUIMIENTO Y MONITOREO DE ACTIVID...,DEL SOLAR PALOMINO PABEL DALMIRO,2022-05-04 15:19:00,2022-05,18,Servicios,Competitivo,1.0,True,0.03
2,2022-05-01 00:00:00-05:00,ocds-dgv273-seacev3-2022-1221-4,2023-12-06 05:33:13.312972-05:00,MUNICIPALIDAD DISTRITAL DE LA CUESTA,PEC-PROC-1-2022-MDLC-1,works,open,2022-05-18 19:36:00-05:00,5.0,51893.73,...,Sin dato,Sin dato,2022-05-18 19:36:00,2022-05,20,Obras,Competitivo,5.0,False,0.00
3,2022-05-01 00:00:00-05:00,ocds-dgv273-seacev3-802121,2023-12-06 05:22:26.998977-05:00,GOBIERNO REGIONAL DE MADRE DE DIOS - HOSPITAL ...,SIE-SIE-3-2022-HSR-2,goods,open,2022-05-04 13:08:00-05:00,NaN,0.00,...,Sin dato,Sin dato,2022-05-04 13:08:00,2022-05,18,Bienes,Competitivo,0.0,False,0.00
4,2022-05-01 00:00:00-05:00,ocds-dgv273-seacev3-804479,2023-12-06 05:26:19.915194-05:00,GOBIERNO REGIONAL DE CAJAMARCA - UNIDAD DE GES...,AS-SM-3-2022-UGEL SAN MIGUEL-2,goods,open,2022-05-12 17:56:00-05:00,6.0,144198.20,...,Sin dato,Sin dato,2022-05-12 17:56:00,2022-05,19,Bienes,Competitivo,6.0,False,0.00
